# Layer-wise Truth Direction Probing: IBM Granite 3.1 8B Instruct (2026)

## Overview & Research Objectives
This notebook evaluates the 2026 enterprise-grade model **IBM Granite 3.1 8B Instruct** (`ibm-granite/granite-3.1-8b-instruct`) across all transformer layers.

By evaluating IBM Granite 8B on the exact same 9 tasks (F0–F5, A1–A3), we perform an apples-to-apples **1:1 architectural comparison against LLaMA 3.1 8B** ($d=4096$, 32 layers):
1. **Generational 8B Comparison (2024 LLaMA 3.1 8B vs. 2026 IBM Granite 8B):** Does 2026 curated training improve factual separation and logical composition?
2. **The Arithmetic Ceiling ($A3$):** Does IBM Granite break past the 3-operation arithmetic limit that LLaMA failed on?
3. **Truth Trajectory ($l/L$):** Where in the network depth does truth emerge in Granite compared to LLaMA and Gemma?

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Environment Setup

In [ ]:
import os
import gc
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Load Model (IBM Granite 3.1 8B Instruct - bfloat16)

In [ ]:
MODEL_ID = "ibm-granite/granite-3.1-8b-instruct"

# Clean up previous memory allocations
try:
    del model
    del tokenizer
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Safe GPU memory ceiling (4.8 GB on GPU 0) with automatic CPU RAM overflow
max_memory = {0: "4.8GiB", "cpu": "30GiB"} if torch.cuda.is_available() else None

print(f"Loading model weights for {MODEL_ID} in bfloat16 (with CPU overflow for 8B scale)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_memory=max_memory
)
model.eval()

config = getattr(model.config, "text_config", model.config)
NUM_HIDDEN_LAYERS = config.num_hidden_layers
HIDDEN_SIZE = config.hidden_size
TOTAL_STATES = NUM_HIDDEN_LAYERS + 1

print(f"Loaded {MODEL_ID} | Layers: {NUM_HIDDEN_LAYERS} | Hidden Dim: {HIDDEN_SIZE}")
if hasattr(model, "hf_device_map"):
    print(f"Device Map Allocation: {model.hf_device_map}")


### Load Datasets (F0–F5, A1–A3)

In [ ]:
candidate_paths = [
    Path("../dataset"),
    Path("dataset"),
    Path("Algoverse_Truth_Directions_Research/dataset"),
    Path("/content/Algoverse_Truth_Directions_Research/dataset"),
    Path("/content/drive/MyDrive/Algoverse_Truth_Directions_Research/dataset")
]

DATA_DIR = next((p for p in candidate_paths if p.exists() and (p / "F0_train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Dataset directory not found. Please check working directory.")

print(f"Using Dataset Directory: {DATA_DIR.resolve()}")

task_names = ["F0", "F1", "F2", "F3", "F4", "F5", "A1", "A2", "A3"]
tasks = {}

for name in task_names:
    train_df = pd.read_csv(DATA_DIR / f"{name}_train.csv")
    test_df = pd.read_csv(DATA_DIR / f"{name}_test.csv")
    tasks[name] = (train_df, test_df)
    print(f"{name:4s} | Train: {len(train_df):4d} | Test: {len(test_df):3d}")

### Layer-wise Activation Extraction

In [ ]:
def activations_all_layers(model, tokenizer, statements, with_chat_template=False, batch_size=4, desc="Extracting"):
    statements = list(statements)
    num_states = getattr(model.config, 'text_config', model.config).num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_states)]

    for i in tqdm(range(0, len(statements), batch_size), desc=desc, leave=False):
        batch = statements[i : i + batch_size]

        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in batch]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True
            ).to(model.device)
        else:
            inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :].detach().cpu().to(torch.float32)
            activations_by_layer[layer_idx].append(final_token)

    return [torch.cat(layer_acts, dim=0) for layer_acts in activations_by_layer]

### Linear Probe Training & Evaluation

In [ ]:
def train_probe(activations, labels, device=torch.device("cuda" if torch.cuda.is_available() else "cpu")):
    (X_train_raw, X_test_raw), (y_train_raw, y_test_raw) = activations, labels

    X_train = X_train_raw.numpy() if isinstance(X_train_raw, torch.Tensor) else np.array(X_train_raw)
    X_test = X_test_raw.numpy() if isinstance(X_test_raw, torch.Tensor) else np.array(X_test_raw)
    y_train = y_train_raw.to_numpy() if hasattr(y_train_raw, "to_numpy") else np.array(y_train_raw)
    y_test = y_test_raw.to_numpy() if hasattr(y_test_raw, "to_numpy") else np.array(y_test_raw)

    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    probe.train()
    for _ in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    
    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc


def train_all_layers(train_acts, test_acts, y_train, y_test, device="cuda"):
    num_layers = len(train_acts)
    layer_results = {}
    for layer_idx in tqdm(range(num_layers), desc="Fitting Layer Probes", leave=False):
        w, train_mean, auroc = train_probe(
            (train_acts[layer_idx], test_acts[layer_idx]),
            (y_train, y_test),
            device=device
        )
        layer_results[layer_idx] = {"auroc": auroc, "weights": w, "train_mean": train_mean}
    return layer_results

### Run Layer-wise Extraction & Probe Training

In [ ]:
all_results_granite8b = {}
output_pkl = Path("layer_results_granite8b.pkl")

for task_name, (train_df, test_df) in tasks.items():
    print(f"\n{'='*20} Processing Task: {task_name} {'='*20}")
    
    train_acts = activations_all_layers(model, tokenizer, train_df["statement"], with_chat_template=False, batch_size=4, desc=f"{task_name} Train Acts")
    test_acts = activations_all_layers(model, tokenizer, test_df["statement"], with_chat_template=False, batch_size=4, desc=f"{task_name} Test Acts")
    
    task_layer_results = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"], device=device)
    all_results_granite8b[task_name] = task_layer_results
    
    top_layers = sorted(task_layer_results.items(), key=lambda x: x[1]["auroc"], reverse=True)[:3]
    print(f"Top Layers for {task_name}:")
    for l_idx, res in top_layers:
        print(f"  -> Layer {l_idx:2d}: AUROC = {res['auroc']:.4f}")
        
    del train_acts, test_acts
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

with open(output_pkl, "wb") as f:
    pickle.dump(all_results_granite8b, f)
print(f"\n✓ Saved IBM Granite 8B layer results to {output_pkl.resolve()}")

### Plot In-Domain Layer AUROC Curves (IBM Granite 8B)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Factual Family (F0 - F5)
ax1 = axes[0]
factual_tasks = ["F0", "F1", "F2", "F3", "F4", "F5"]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(factual_tasks)))

for task, color in zip(factual_tasks, colors):
    if task in all_results_granite8b:
        layers = sorted(all_results_granite8b[task].keys())
        aurocs = [all_results_granite8b[task][l]["auroc"] for l in layers]
        ax1.plot(layers, aurocs, marker='o', label=task, color=color, linewidth=2, markersize=4)

ax1.set_title("IBM Granite 8B (2026): Factual Family (F0 - F5)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Layer Index", fontsize=11)
ax1.set_ylabel("In-Domain Test AUROC", fontsize=11)
ax1.set_ylim(0.45, 1.02)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance (0.5)')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='lower right', frameon=True)

# 2. Arithmetic Family (A1 - A3)
ax2 = axes[1]
arith_tasks = ["A1", "A2", "A3"]
arith_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for task, color in zip(arith_tasks, arith_colors):
    if task in all_results_granite8b:
        layers = sorted(all_results_granite8b[task].keys())
        aurocs = [all_results_granite8b[task][l]["auroc"] for l in layers]
        ax2.plot(layers, aurocs, marker='s', label=task, color=color, linewidth=2, markersize=4)

ax2.set_title("IBM Granite 8B (2026): Arithmetic Family (A1 - A3)", fontsize=13, fontweight='bold')
ax2.set_xlabel("Layer Index", fontsize=11)
ax2.set_ylabel("In-Domain Test AUROC", fontsize=11)
ax2.set_ylim(0.45, 1.02)
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance (0.5)')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()

### 4-Way Relative Depth Comparison: Gemma 4 (2B) vs. LLaMA 3.2 (3B) vs. LLaMA 3.1 (8B) vs. IBM Granite (8B)

In [ ]:
# Load existing pickle files
gemma_pkl = Path("layer_results_gemma4.pkl") if Path("layer_results_gemma4.pkl").exists() else Path("validation_code/layer_results_gemma4.pkl")
llama3b_pkl = Path("layer_results_llama3_2_3b.pkl") if Path("layer_results_llama3_2_3b.pkl").exists() else Path("validation_code/layer_results_llama3_2_3b.pkl")
llama8b_pkl = Path("layer_results.pkl") if Path("layer_results.pkl").exists() else Path("validation_code/layer_results.pkl")

all_results_gemma4 = pickle.load(open(gemma_pkl, "rb")) if gemma_pkl.exists() else {}
all_results_llama3b = pickle.load(open(llama3b_pkl, "rb")) if llama3b_pkl.exists() else {}
all_results_llama8b = pickle.load(open(llama8b_pkl, "rb")) if llama8b_pkl.exists() else {}

selected_tasks = ["F0", "F1", "F2", "F4", "A1"]
fig, axes = plt.subplots(1, len(selected_tasks), figsize=(24, 4.5), sharey=True)

for idx, task in enumerate(selected_tasks):
    ax = axes[idx]
    
    # 1. Gemma 4 Edge 2B (2026 Small)
    if task in all_results_gemma4:
        g_layers = sorted(all_results_gemma4[task].keys())
        g_rel = [l / max(g_layers) for l in g_layers]
        g_auroc = [all_results_gemma4[task][l]["auroc"] for l in g_layers]
        ax.plot(g_rel, g_auroc, marker='o', label='Gemma 4 (2B, 2026)', color='#e63946', linewidth=2)
        
    # 2. LLaMA 3.2 3B (2024 Small)
    if task in all_results_llama3b:
        l3b_layers = sorted(all_results_llama3b[task].keys())
        l3b_rel = [l / max(l3b_layers) for l in l3b_layers]
        l3b_auroc = [all_results_llama3b[task][l]["auroc"] for l in l3b_layers]
        ax.plot(l3b_rel, l3b_auroc, marker='s', linestyle='-.', label='LLaMA 3.2 (3B, 2024)', color='#2a9d8f', linewidth=2)
        
    # 3. LLaMA 3.1 8B (2024 Large Baseline)
    if task in all_results_llama8b:
        l8b_layers = sorted(all_results_llama8b[task].keys())
        l8b_rel = [l / max(l8b_layers) for l in l8b_layers]
        l8b_auroc = [all_results_llama8b[task][l]["auroc"] for l in l8b_layers]
        ax.plot(l8b_rel, l8b_auroc, marker='^', linestyle='--', label='LLaMA 3.1 (8B, 2024)', color='#1d3557', linewidth=2)
        
    # 4. IBM Granite 8B (2026 Large)
    if task in all_results_granite8b:
        gr_layers = sorted(all_results_granite8b[task].keys())
        gr_rel = [l / max(gr_layers) for l in gr_layers]
        gr_auroc = [all_results_granite8b[task][l]["auroc"] for l in gr_layers]
        ax.plot(gr_rel, gr_auroc, marker='D', linestyle='-', label='IBM Granite (8B, 2026)', color='#8338ec', linewidth=2)
        
    ax.set_title(f"Task {task}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Relative Depth (l / L)", fontsize=10)
    if idx == 0:
        ax.set_ylabel("In-Domain AUROC", fontsize=11)
    ax.set_ylim(0.45, 1.02)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='lower right', fontsize=8)

plt.suptitle("4-Way Truth Trajectory: Gemma 4 (2B) vs. LLaMA 3.2 (3B) vs. LLaMA 3.1 (8B) vs. IBM Granite (8B)", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### 4-Way Comparative Findings & Summary

| Task | Description | Gemma 4 (2B) AUROC | LLaMA 3.2 (3B) AUROC | LLaMA 3.1 (8B) AUROC | IBM Granite (8B) AUROC |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **F0** | Atomic Facts | 0.8554 | 0.9993 | 0.9998 | *TBD* |
| **F1** | Negations | 0.8807 | 1.0000 | 0.9999 | *TBD* |
| **F2** | Conjunctions | 0.7814 | 0.9945 | 0.9991 | *TBD* |
| **F3** | 2-City Counting | 0.6199 | 0.9388 | 0.9779 | *TBD* |
| **F4** | 5-City Counting | 0.6017 | 0.8531 | 0.8707 | *TBD* |
| **F5** | Dual Counting | 0.5666 | 0.7545 | 0.8008 | *TBD* |
| **A1** | 1-Op Arithmetic | 0.5622 | 0.9988 | 0.9992 | *TBD* |
| **A2** | Two-Op Arithmetic | 0.5298 | 0.8985 | 0.8848 | *TBD* |
| **A3** | Three-Op Arithmetic | 0.5461 | 0.5978 | 0.5972 | *TBD* |

---

### Key Takeaways
1. **1:1 8B Architecture Comparison (LLaMA 3.1 8B vs. IBM Granite 8B):** 
2. **Scale vs. Architecture Trajectory:** 
3. **Arithmetic Circuit Limits:** 